In [ ]:
# ============================================================================
# CATBOOST PIPELINE: THERAPY OPTIMIZATION FOR SUICIDE RISK REDUCTION
# ============================================================================
# CatBoost advantages for this analysis:
# - Native categorical feature handling (therapist names, locations, etc.)
# - Built-in class weight balancing
# - Robust to overfitting with ordered boosting
# - Fast GPU support if available
# ============================================================================

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(123)

print("=" * 90)
print("CATBOOST ANALYSIS: RISK IMPROVEMENT PREDICTION & THERAPY OPTIMIZATION")
print("=" * 90)

# ============================================================================
# PART 1: LOAD DATA AND CREATE OUTCOME
# ============================================================================
print("\n[STEP 1] Loading data and creating outcome variable...")

# Load data
data = pd.read_csv('../data.csv')
print(f"  Loaded {len(data)} patients")

# Create outcome variable
risk_order = {"Low": 1, "Moderate": 2, "High": 3}
data['risk_initial_num'] = data['risk_level_initial'].map(risk_order)
data['risk_first_num'] = data['risk_level_srs_first'].map(risk_order)
data['improve'] = (data['risk_first_num'] < data['risk_initial_num']).astype(int)
data['risk_change'] = data['risk_initial_num'] - data['risk_first_num']

improvement_rate = data['improve'].mean()
print(f"  Overall improvement rate: {improvement_rate*100:.1f}%")

# Risk transition matrix
risk_transitions = pd.crosstab(data['risk_level_initial'], 
                               data['risk_level_srs_first'], 
                               margins=True)
print("\nRisk transitions:")
print(risk_transitions)

# ============================================================================
# PART 2: FEATURE ENGINEERING
# ============================================================================
print("\n[STEP 2] Feature engineering...")

# Define feature groups
patient_clinical_features = [
    'total_score', 'risk_high_initial', 'deterrents_month', 'what_sort_of_reasons',
    'duration_month', 'adolescent', 'are_there_things', 'frequency_month',
    'when_you_have_the_thoughts_how_long_do_they_last',
    'how_many_times_have_you_had_these_thoughts', 'male', 'dx_group'
]

# Add diagnosis/symptom features
diagnosis_features = [col for col in data.columns if col.startswith('current_and_past_psychiatric_diagnoses_')]
symptom_features = [col for col in data.columns if col.startswith('presenting_symptoms_')]
family_features = [col for col in data.columns if col.startswith('family_history_')]
stressor_features = [col for col in data.columns if col.startswith('precipitants_stressors_')]
protective_features = [col for col in data.columns if 'protective_factors_' in col]

patient_clinical_features.extend(diagnosis_features + symptom_features + 
                                 family_features + stressor_features + protective_features)

# Therapy features
therapy_features = ['act', 'cbt', 'dbt', 'motivational_interviewing',
                   'mindfulness', 'stages_of_change', 'family_systems']
therapy_features = [f for f in therapy_features if f in data.columns]

# Propensity scores
propensity_features = [col for col in data.columns if col.startswith('prop_')]

# Treatment context
treatment_context_features = ['therapy_duration_category', 'delivery_method',
                             'session_mode']
treatment_context_features = [f for f in treatment_context_features if f in data.columns]

# Organizational features (will be treated as categorical)
categorical_features = ['therapist_name', 'location', 'program', 'dx_group',
                       'therapy_duration_category', 'delivery_method', 'session_mode',
                       'pn_month', 'pn_time_block', 'pn_year']
categorical_features = [f for f in categorical_features if f in data.columns]

# ============================================================================
# PART 3: CREATE INTERACTION FEATURES
# ============================================================================
print("\n[STEP 3] Creating interaction features...")

interaction_features = []

for therapy in therapy_features:
    # Interaction with risk level
    feat_name = f'{therapy}_x_high_risk'
    data[feat_name] = data[therapy] * data['risk_high_initial']
    interaction_features.append(feat_name)
    
    # Interaction with total score
    if 'total_score' in data.columns:
        feat_name = f'{therapy}_x_total_score'
        data[feat_name] = data[therapy] * data['total_score']
        interaction_features.append(feat_name)
    
    # Interaction with age
    if 'adolescent' in data.columns:
        feat_name = f'{therapy}_x_adolescent'
        data[feat_name] = data[therapy] * data['adolescent']
        interaction_features.append(feat_name)

# Therapy combinations
data['n_therapies'] = data[therapy_features].sum(axis=1)
data['high_therapy_intensity'] = (data['n_therapies'] >= 3).astype(int)

if all(t in therapy_features for t in ['cbt', 'dbt']):
    data['has_cbt_dbt'] = ((data['cbt'] == 1) & (data['dbt'] == 1)).astype(int)
    
if all(t in therapy_features for t in ['motivational_interviewing', 'cbt']):
    data['has_mi_cbt'] = ((data['motivational_interviewing'] == 1) & 
                          (data['cbt'] == 1)).astype(int)

combination_features = ['n_therapies', 'high_therapy_intensity']
if 'has_cbt_dbt' in data.columns:
    combination_features.append('has_cbt_dbt')
if 'has_mi_cbt' in data.columns:
    combination_features.append('has_mi_cbt')

print(f"  Created {len(interaction_features)} interaction features")
print(f"  Created {len(combination_features)} combination features")

# ============================================================================
# PART 4: TEMPORAL TRAIN-TEST SPLIT
# ============================================================================
print("\n[STEP 4] Creating temporal split (80/20)...")

# Sort by admission date
data['admission_date'] = pd.to_datetime(data['admission_date'])
data = data.sort_values('admission_date').reset_index(drop=True)

# Split at 80%
split_point = int(len(data) * 0.8)
train_data = data.iloc[:split_point].copy()
test_data = data.iloc[split_point:].copy()

print(f"  Training set: n={len(train_data)} ({train_data['improve'].mean()*100:.1f}% improved)")
print(f"  Test set: n={len(test_data)} ({test_data['improve'].mean()*100:.1f}% improved)")
print(f"  Training dates: {train_data['admission_date'].min()} to {train_data['admission_date'].max()}")
print(f"  Test dates: {test_data['admission_date'].min()} to {test_data['admission_date'].max()}")

# ============================================================================
# PART 5: PREPARE FEATURES FOR CATBOOST
# ============================================================================
print("\n[STEP 5] Preparing features for CatBoost...")

# Combine all features
all_features = list(set(
    patient_clinical_features + therapy_features + propensity_features +
    treatment_context_features + categorical_features + 
    interaction_features + combination_features
))

# Remove outcome and date columns
exclude_cols = ['improve', 'risk_change', 'admission_date', 'discharge_date',
                'risk_level_initial', 'risk_level_srs_first', 
                'risk_initial_num', 'risk_first_num', 'master_id', 'row_id']
all_features = [f for f in all_features if f in data.columns and f not in exclude_cols]

print(f"  Total features: {len(all_features)}")
print(f"  Categorical features: {len([f for f in all_features if f in categorical_features])}")
print(f"  Numeric features: {len([f for f in all_features if f not in categorical_features])}")

# Prepare data
X_train = train_data[all_features]
X_test = test_data[all_features]
y_train = train_data['improve']
y_test = test_data['improve']

# Get categorical feature indices
cat_feature_indices = [i for i, col in enumerate(all_features) if col in categorical_features]

# ============================================================================
# PART 6: HYPERPARAMETER TUNING
# ============================================================================
print("\n[STEP 6] Hyperparameter tuning for CatBoost...")

# Calculate class weight
class_weight = len(y_train) / (2 * np.bincount(y_train))
scale_pos_weight = class_weight[1] / class_weight[0]
print(f"  Class weight (scale_pos_weight): {scale_pos_weight:.3f}")

# Define parameter grid
param_grid = {
    'iterations': [300, 500, 700, 1000],
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7, 9],
    'border_count': [32, 64, 128, 255],
    'subsample': [0.66, 0.8, 1.0],
}

# Create base model
base_model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    scale_pos_weight=scale_pos_weight,
    cat_features=cat_feature_indices,
    random_seed=123,
    verbose=False,
    early_stopping_rounds=50,
    use_best_model=True
)

# Quick search with fewer iterations for speed
print("  Running randomized search (this may take a few minutes)...")
search = RandomizedSearchCV(
    base_model,
    param_grid,
    n_iter=50,  # Number of parameter combinations to try
    scoring='roc_auc',
    cv=TimeSeriesSplit(n_splits=3),  # Temporal CV
    random_state=123,
    n_jobs=-1,
    verbose=0
)

search.fit(X_train, y_train, 
          eval_set=(X_test, y_test),
          verbose=False)

best_params = search.best_params_
print(f"\n  Best CV AUC: {search.best_score_:.4f}")
print(f"  Best parameters: {best_params}")

# ============================================================================
# PART 7: TRAIN FINAL MODEL
# ============================================================================
print("\n[STEP 7] Training final CatBoost model...")

# Train final model with best parameters
final_model = CatBoostClassifier(
    **best_params,
    loss_function='Logloss',
    eval_metric='AUC',
    scale_pos_weight=scale_pos_weight,
    cat_features=cat_feature_indices,
    random_seed=123,
    verbose=100,
    early_stopping_rounds=50,
    use_best_model=True
)

# Create pools for efficient training
train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_feature_indices)

# Train
final_model.fit(train_pool, eval_set=test_pool)

# Get predictions
pred_train = final_model.predict_proba(X_train)[:, 1]
pred_test = final_model.predict_proba(X_test)[:, 1]

# Calculate metrics
auc_train = roc_auc_score(y_train, pred_train)
auc_test = roc_auc_score(y_test, pred_test)
brier_score = brier_score_loss(y_test, pred_test)

print(f"\n  Training AUC: {auc_train:.4f}")
print(f"  Test AUC: {auc_test:.4f}")
print(f"  Brier Score: {brier_score:.4f}")
print(f"  Best iteration: {final_model.best_iteration_}")

# Confusion matrix at optimal threshold
fpr, tpr, thresholds = roc_curve(y_test, pred_test)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

pred_test_binary = (pred_test > optimal_threshold).astype(int)
cm = confusion_matrix(y_test, pred_test_binary)

print("\nConfusion Matrix:")
print(cm)

if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    
    print(f"\nTest Set Performance (threshold = {optimal_threshold:.3f}):")
    print(f"  Sensitivity: {sensitivity*100:.1f}%")
    print(f"  Specificity: {specificity*100:.1f}%")
    print(f"  PPV: {ppv*100:.1f}%")
    print(f"  NPV: {npv*100:.1f}%")

# ============================================================================
# PART 8: FEATURE IMPORTANCE
# ============================================================================
print("\n[STEP 8] Analyzing feature importance...")

# Get CatBoost feature importance
feature_importance = pd.DataFrame({
    'feature': all_features,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 features by importance:")
print(feature_importance.head(20))

# Categorize by group
def categorize_feature(feat):
    if feat in therapy_features:
        return "Therapy Received"
    elif feat.startswith('prop_'):
        return "Propensity Score"
    elif feat in categorical_features:
        return "Therapist/Org"
    elif feat in interaction_features:
        return "Treatment Interactions"
    elif feat in combination_features:
        return "Therapy Combinations"
    elif feat in treatment_context_features:
        return "Treatment Context"
    else:
        return "Patient Clinical"

feature_importance['group'] = feature_importance['feature'].apply(categorize_feature)
group_importance = feature_importance.groupby('group')['importance'].agg(['sum', 'mean', 'count'])
group_importance = group_importance.sort_values('sum', ascending=False)

print("\nFeature Importance by Group:")
print(group_importance)

# ============================================================================
# PART 9: DOUBLY ROBUST ESTIMATION (FIXED)
# ============================================================================
print("\n[STEP 9] Performing doubly robust estimation...")

dr_results = {}

for therapy in therapy_features:
    prop_col = f'prop_{therapy.replace("_", "")}'  # Handle underscores in names
    if prop_col not in X_train.columns:
        prop_col = f'prop_{therapy[:3]}'  # Try abbreviated version
    
    if prop_col in X_train.columns:
        T = X_train[therapy].values
        e = X_train[prop_col].values
        e = np.clip(e, 0.01, 0.99)  # Trim propensity scores
        
        n_treated = T.sum()
        n_control = len(T) - n_treated
        
        if n_treated >= 20 and n_control >= 20:
            # Fit separate models
            X_without_therapy = X_train.drop(columns=[therapy])
            
            # Get NEW categorical indices after dropping the therapy column
            features_without_therapy = [f for f in all_features if f != therapy]
            cat_indices_without_therapy = [i for i, col in enumerate(features_without_therapy) 
                                          if col in categorical_features]
            
            # Treated model with corrected categorical features
            model_t = CatBoostClassifier(
                **best_params, 
                verbose=False, 
                random_seed=123,
                cat_features=cat_indices_without_therapy  # Specify categorical features
            )
            model_t.fit(X_without_therapy[T == 1], y_train[T == 1])
            
            # Control model with corrected categorical features
            model_c = CatBoostClassifier(
                **best_params, 
                verbose=False, 
                random_seed=123,
                cat_features=cat_indices_without_therapy  # Specify categorical features
            )
            model_c.fit(X_without_therapy[T == 0], y_train[T == 0])
            
            # Predict potential outcomes
            mu_1 = model_t.predict_proba(X_without_therapy)[:, 1]
            mu_0 = model_c.predict_proba(X_without_therapy)[:, 1]
            
            # AIPW estimator
            tau_i = (mu_1 - mu_0 + 
                    T * (y_train - mu_1) / e - 
                    (1 - T) * (y_train - mu_0) / (1 - e))
            
            ate = tau_i.mean()
            se = tau_i.std() / np.sqrt(len(tau_i))
            
            dr_results[therapy] = {
                'ate': ate,
                'se': se,
                'ci_lower': ate - 1.96 * se,
                'ci_upper': ate + 1.96 * se,
                'n_treated': n_treated,
                'n_control': n_control
            }
            
            print(f"  {therapy}: ATE = {ate:.3f} (95% CI: {ate-1.96*se:.3f} to {ate+1.96*se:.3f}), n={n_treated}")
        else:
            print(f"  {therapy}: Insufficient sample size (n_treated={n_treated}, n_control={n_control})")

# ============================================================================
# PART 10: COUNTERFACTUAL ANALYSIS
# ============================================================================
print("\n[STEP 10] Computing personalization gains...")

# Get observed therapy combinations
combo_matrix = X_test[therapy_features].values
combo_strings = [''.join(map(str, row)) for row in combo_matrix.astype(int)]
combo_counts = pd.Series(combo_strings).value_counts()
top_combos = combo_counts.head(50).index.tolist()

print(f"  Evaluating {len(top_combos)} therapy combinations")

personalization_gains = np.zeros(len(X_test))
optimal_combos = np.zeros((len(X_test), len(therapy_features)))

for i in tqdm(range(len(X_test)), desc="Computing gains"):
    patient_features = X_test.iloc[i].copy()
    baseline_prob = pred_test[i]
    
    best_prob = baseline_prob
    best_combo = combo_matrix[i]
    
    for combo_str in top_combos:
        combo_vals = np.array([int(c) for c in combo_str])
        
        if len(combo_vals) != len(therapy_features):
            continue
            
        # Create counterfactual
        cf_features = patient_features.copy()
        
        # Update therapy values
        for j, therapy in enumerate(therapy_features):
            cf_features[therapy] = combo_vals[j]
            
        # Update interaction features
        for therapy in therapy_features:
            if f'{therapy}_x_high_risk' in cf_features.index:
                cf_features[f'{therapy}_x_high_risk'] = (
                    cf_features[therapy] * cf_features['risk_high_initial']
                )
            if f'{therapy}_x_total_score' in cf_features.index:
                cf_features[f'{therapy}_x_total_score'] = (
                    cf_features[therapy] * cf_features.get('total_score', 0)
                )
                
        # Update combination features
        cf_features['n_therapies'] = combo_vals.sum()
        if 'high_therapy_intensity' in cf_features.index:
            cf_features['high_therapy_intensity'] = int(combo_vals.sum() >= 3)
            
        # Predict counterfactual outcome
        cf_prob = final_model.predict_proba(cf_features.values.reshape(1, -1))[:, 1][0]
        
        if cf_prob > best_prob:
            best_prob = cf_prob
            best_combo = combo_vals
    
    personalization_gains[i] = best_prob - baseline_prob
    optimal_combos[i] = best_combo

# Summary statistics
mean_gain = personalization_gains.mean()
median_gain = np.median(personalization_gains)
pct_benefit = (personalization_gains > 0.01).mean() * 100
pct_large_benefit = (personalization_gains > 0.05).mean() * 100

print(f"\nPersonalization Gain Summary:")
print(f"  Mean gain: {mean_gain:.3f} ({mean_gain/pred_test.mean()*100:.1f}% relative)")
print(f"  Median gain: {median_gain:.3f}")
print(f"  Patients who would benefit (>1% gain): {pct_benefit:.1f}%")
print(f"  Patients with substantial benefit (>5% gain): {pct_large_benefit:.1f}%")
print(f"  Maximum gain: {personalization_gains.max():.3f}")
print(f"  NNT to prevent one non-improvement: {1/mean_gain if mean_gain > 0 else np.inf:.0f}")

# ============================================================================
# PART 11: SAVE RESULTS
# ============================================================================
print("\n[STEP 11] Saving results...")

# Create results dictionary
results = {
    'model': final_model,
    'performance': {
        'auc_train': auc_train,
        'auc_test': auc_test,
        'brier_score': brier_score,
        'sensitivity': sensitivity if 'sensitivity' in locals() else None,
        'specificity': specificity if 'specificity' in locals() else None,
        'ppv': ppv if 'ppv' in locals() else None,
        'npv': npv if 'npv' in locals() else None
    },
    'feature_importance': feature_importance,
    'group_importance': group_importance,
    'dr_results': dr_results,
    'personalization': {
        'gains': personalization_gains,
        'optimal_combos': optimal_combos,
        'mean_gain': mean_gain,
        'pct_benefit': pct_benefit
    }
}

# Save to pickle
import pickle
with open('catboost_results.pkl', 'wb') as f:
    pickle.dump(results, f)

# Save CSVs
feature_importance.to_csv('feature_importance.csv', index=False)
pd.DataFrame(dr_results).T.to_csv('therapy_associations.csv')

print("\nFiles saved:")
print("  - catboost_results.pkl")
print("  - feature_importance.csv")
print("  - therapy_associations.csv")

print("\n" + "="*90)
print("ANALYSIS COMPLETE")
print("="*90)

CATBOOST ANALYSIS: RISK IMPROVEMENT PREDICTION & THERAPY OPTIMIZATION

[STEP 1] Loading data and creating outcome variable...
  Loaded 5986 patients
  Overall improvement rate: 69.3%

Risk transitions:
risk_level_srs_first  High   Low  Moderate   All
risk_level_initial                              
High                   357   268      1012  1637
Moderate               181  2869      1299  4349
All                    538  3137      2311  5986

[STEP 2] Feature engineering...

[STEP 3] Creating interaction features...
  Created 21 interaction features
  Created 4 combination features

[STEP 4] Creating temporal split (80/20)...
  Training set: n=4788 (67.0% improved)
  Test set: n=1198 (78.6% improved)
  Training dates: 2020-10-06 00:00:00 to 2023-05-04 00:00:00
  Test dates: 2023-05-04 00:00:00 to 2023-12-19 00:00:00

[STEP 5] Preparing features for CatBoost...
  Total features: 107
  Categorical features: 10
  Numeric features: 97

[STEP 6] Hyperparameter tuning for CatBoost...
  Clas

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=31]="Sherman Oaks OP, CA 3009": Cannot convert 'Sherman Oaks OP, CA 3009' to float